# Replace hand-typed tracker list with Disconnect.me's verified list

Source: https://github.com/disconnectme/disconnect-tracking-protection
This is the SAME dataset used by Firefox's built-in tracking protection,
licensed under Creative Commons (CC BY-NC-SA 4.0) — free to use for
a non-commercial student project, with attribution.

Structure of the file (services.json):
{
  "categories": {
    "Advertising": [ {"CompanyName": {"homepage_url": ["domain1.com", "domain2.com"]}}, ... ],
    "Analytics": [ ... ],
    "ConsentManagers": [ ... ],   <- this is exactly our CMP detection category
    "Email": [ ... ],
    ...
  }
}

# Step 1 — Download the list

In [1]:
import requests
import json

DISCONNECT_URL = 'https://raw.githubusercontent.com/disconnectme/disconnect-tracking-protection/master/services.json'

response = requests.get(DISCONNECT_URL)
disconnect_data = response.json()

print('Categories found:')
print(list(disconnect_data['categories'].keys()))

Categories found:
['Email', 'EmailAggressive', 'Advertising', 'Content', 'Analytics', 'FingerprintingInvasive', 'FingerprintingGeneral', 'Anti-fraud', 'Social', 'ConsentManagers', 'Cryptomining']


# Step 2 — Save a local copy
Important for reproducibility — the live list can change over time,
so save the exact version you used for your project.

In [2]:
with open('../raw_data/disconnect_services.json', 'w') as f:
    json.dump(disconnect_data, f, indent=2)

print('Saved local copy: disconnect_services.json')

Saved local copy: disconnect_services.json


# Step 3 — Build a flat domain -> category lookup dictionary
Converts the nested JSON into a simple dict: {domain: category}
This is much faster to check against during scraping than searching
the nested structure every time.

In [3]:
def build_domain_lookup(disconnect_data):
    """
    Flattens the Disconnect services.json into {domain: category}.
    Also returns {domain: company_name} for richer reporting later.
    """
    domain_to_category = {}
    domain_to_company = {}
    
    for category, services in disconnect_data['categories'].items():
        for service in services:
            for company_name, urls_dict in service.items():
                for homepage, domains in urls_dict.items():
                    for domain in domains:
                        domain_to_category[domain] = category
                        domain_to_company[domain] = company_name
    
    return domain_to_category, domain_to_company

domain_to_category, domain_to_company = build_domain_lookup(disconnect_data)

print(f'Total tracked domains loaded: {len(domain_to_category)}')
print()
print('Sample entries:')
for domain in list(domain_to_category.keys())[:10]:
    print(f'  {domain} -> {domain_to_category[domain]} ({domain_to_company[domain]})')

Total tracked domains loaded: 4443

Sample entries:
  10web.io -> Email (10Web)
  4dem.it -> Email (4dem)
  8d8.biz -> Email (8d8.biz)
  open.mkt10008.com -> Email (Acoustic)
  open.mkt10039.com -> Email (Acoustic)
  open.mkt10049.com -> Email (Acoustic)
  open.mkt10067.com -> Email (Acoustic)
  open.mkt10114.com -> Email (Acoustic)
  open.mkt10153.com -> Email (Acoustic)
  open.mkt10663.com -> Email (Acoustic)


# Step 4 — Verify known trackers are present
Quick sanity check — confirm common trackers we expect ARE in the list.

In [4]:
check_domains = [
    'google-analytics.com', 'doubleclick.net', 'facebook.net',
    'criteo.com', 'hotjar.com', 'segment.io'
]

for d in check_domains:
    category = domain_to_category.get(d, 'NOT FOUND')
    print(f'{d}: {category}')

google-analytics.com: FingerprintingGeneral
doubleclick.net: FingerprintingGeneral
facebook.net: Social
criteo.com: Advertising
hotjar.com: Analytics
segment.io: Analytics


# Step 5 — Check what categories exist for Consent Managers specifically
This confirms whether the category is literally called 'ConsentManagers'
and shows you which CMPs are in Disconnect's list.

In [5]:
cmp_categories = [cat for cat in disconnect_data['categories'].keys() if 'consent' in cat.lower()]
print('Consent-related category names found:', cmp_categories)
print()

if cmp_categories:
    cmp_cat_name = cmp_categories[0]
    cmp_domains = [d for d, c in domain_to_category.items() if c == cmp_cat_name]
    print(f'Total CMP-related domains in this category: {len(cmp_domains)}')
    print('Examples:', cmp_domains[:10])

Consent-related category names found: ['ConsentManagers']

Total CMP-related domains in this category: 14
Examples: ['axept.io', 'privacy-center.org', 'privacy-mgmt.com', 'summerhamster.com', 'cookielaw.org', 'onetrust.com', 'onetrust.io', 'osano.com', 'transcend.io', 'trustarc.com']


# Step 6 — New, verified versions of your detection functions
These REPLACE the hand-typed KNOWN_AD_NETWORKS list from before.

In [6]:
def classify_domain(domain):
    """
    Checks a domain against the Disconnect list.
    Returns (is_tracker: bool, category: str or None, company: str or None)
    Handles subdomains — e.g. 'www.google-analytics.com' should still match.
    """
    for known_domain, category in domain_to_category.items():
        if known_domain in domain:
            return True, category, domain_to_company.get(known_domain)
    return False, None, None

def is_known_ad_network(domain):
    """Drop-in replacement for the old function — now backed by real data."""
    is_tracker, category, _ = classify_domain(domain)
    return is_tracker and category in ('Advertising', 'Analytics')

def is_cmp_domain(domain):
    """NEW — lets you detect CMPs directly from network requests too,
    as a cross-check against your BigQuery technologies data."""
    is_tracker, category, _ = classify_domain(domain)
    return is_tracker and category in cmp_categories

# Step 7 — Test the new functions

In [7]:
test_domains = [
    'google-analytics.com', 'doubleclick.net', 'fonts.googleapis.com',
    'cdn.jsdelivr.net', 'criteo.com'
]

for d in test_domains:
    is_tracker, category, company = classify_domain(d)
    print(f'{d}: is_tracker={is_tracker}, category={category}, company={company}')

google-analytics.com: is_tracker=True, category=FingerprintingGeneral, company=Google
doubleclick.net: is_tracker=True, category=FingerprintingGeneral, company=Google
fonts.googleapis.com: is_tracker=True, category=Content, company=Google
cdn.jsdelivr.net: is_tracker=True, category=Content, company=Volentio JSD
criteo.com: is_tracker=True, category=Advertising, company=Criteo
